# HUD Qualified Census Tracts → LAPD Precinct Assignment

Assigns each HUD QCT census tract centroid to an LAPD precinct and writes
`data/census_indicators/qct_by_prec.csv` with columns `PREC, qct_count`.

Logic:
- Fetch LA County census tract centroids from the Census TIGER REST API (layer 6, paginated).
- Filter the HUD QCT file to rows where `HUD_QCT == "Yes"`.
- For each qualifying tract centroid, assign it to the precinct whose circular region
  contains it (using haversine distance ≤ radius). If no circle contains it, assign to nearest.

In [1]:
import math
import requests
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")
QCT_PATH = DATA_DIR / "census_indicators" / "HUD_Qualified_Census_Tracts_2025.csv"
PREC_PATH = DATA_DIR / "LAPD" / "lapd_precincts_combined.csv"
OUT_PATH  = DATA_DIR / "census_indicators" / "qct_by_prec.csv"

TIGER_URL = (
    "https://tigerweb.geo.census.gov/arcgis/rest/services/"
    "TIGERweb/tigerWMS_Census2020/MapServer/6/query"
)

In [2]:
# --- 1. Fetch LA County census tract centroids (paginated) ---

def fetch_tract_centroids(url: str, page_size: int = 500) -> pd.DataFrame:
    records = []
    offset = 0
    while True:
        resp = requests.get(
            url,
            params={
                "where": "STATE='06' AND COUNTY='037'",
                "outFields": "GEOID,CENTLAT,CENTLON",
                "returnGeometry": "false",
                "f": "json",
                "resultRecordCount": page_size,
                "resultOffset": offset,
            },
            timeout=30,
        )
        data = resp.json()
        features = data.get("features", [])
        records.extend(f["attributes"] for f in features)
        if not data.get("exceededTransferLimit") or not features:
            break
        offset += page_size
    return pd.DataFrame(records)

tiger = fetch_tract_centroids(TIGER_URL)
tiger["GEOID"] = tiger["GEOID"].astype(str).str.zfill(11)
tiger["CENTLAT"] = tiger["CENTLAT"].astype(float)
tiger["CENTLON"] = tiger["CENTLON"].astype(float)
print(f"Fetched {len(tiger)} LA County census tracts")
tiger.head(3)

Fetched 2498 LA County census tracts


,GEOID,CENTLAT,CENTLON
0,06037408722,33.970025,-117.902348
1,06037311602,34.151978,-118.345143
2,06037980024,34.176678,-118.488620


In [3]:
# --- 2. Load HUD QCT, filter to qualified tracts, merge centroids ---

qct = pd.read_csv(QCT_PATH)
print(f"Total tracts in HUD file: {len(qct)}")
print(f"HUD_QCT == 'Yes': {(qct['HUD_QCT'] == 'Yes').sum()}")

qct = qct[qct["HUD_QCT"] == "Yes"][["GEOID"]].copy()
qct["GEOID"] = qct["GEOID"].astype(str).str.zfill(11)

qct = qct.merge(tiger, on="GEOID", how="left")
missing = qct["CENTLAT"].isna().sum()
print(f"Matched {len(qct) - missing}/{len(qct)} qualified tracts to centroids")
qct = qct.dropna(subset=["CENTLAT", "CENTLON"])
qct.head(3)

Total tracts in HUD file: 2495
HUD_QCT == 'Yes': 584
Matched 584/584 qualified tracts to centroids


,GEOID,CENTLAT,CENTLON
0,06037294610,33.787324,-118.258197
1,06037242100,33.945630,-118.232366
2,06037239202,33.985678,-118.264977


In [4]:
# --- 3. Load precinct stations and compute circle radii ---

prec = pd.read_csv(PREC_PATH, usecols=["PREC", "DIVISION", "lat", "lon", "Shape__Area"])

# radius uses same formula as _circle_polygon in map_builder.py
prec["radius_m"] = (prec["Shape__Area"] * 0.0929 / math.pi).pow(0.5)

print(prec[["PREC", "DIVISION", "radius_m"]].to_string(index=False))

 PREC         DIVISION    radius_m
    5           HARBOR 5138.412703
   18        SOUTHEAST 2778.873905
   12      77TH STREET 3056.668071
   14          PACIFIC 4606.573721
    3        SOUTHWEST 3186.294609
   13           NEWTON 2838.032407
    1          CENTRAL 2010.911764
    8 WEST LOS ANGELES 7303.147742
    4       HOLLENBECK 3578.434654
    7         WILSHIRE 3109.935167
   20          OLYMPIC 2271.062916
    2          RAMPART 2136.732139
    6        HOLLYWOOD 3315.951945
   11        NORTHEAST 4911.024673
   15  NORTH HOLLYWOOD 4331.948746
    9         VAN NUYS 3805.360021
   10      WEST VALLEY 5262.170955
   21          TOPANGA 5184.688399
   16         FOOTHILL 6193.944102
   17       DEVONSHIRE 6310.839190
   19          MISSION 4548.334777


In [5]:
# --- 4. Haversine helper and precinct assignment ---

def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    R = 6_371_000
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2))
         * math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))

stations = [
    (int(row.PREC), float(row.lat), float(row.lon), float(row.radius_m))
    for row in prec.itertuples(index=False)
]

def assign_precinct(tract_lat: float, tract_lon: float) -> int:
    dists = [
        (prec_id, haversine_m(tract_lat, tract_lon, slat, slon), r)
        for prec_id, slat, slon, r in stations
    ]
    in_circle = [(prec_id, d) for prec_id, d, r in dists if d <= r]
    if in_circle:
        return min(in_circle, key=lambda x: x[1])[0]
    return min(dists, key=lambda x: x[1])[0]

qct["PREC"] = qct.apply(lambda row: assign_precinct(row["CENTLAT"], row["CENTLON"]), axis=1)
print(qct[["GEOID", "CENTLAT", "CENTLON", "PREC"]].head(5))

         GEOID    CENTLAT     CENTLON  PREC
0  06037294610  33.787324 -118.258197     5
1  06037242100  33.945630 -118.232366    18
2  06037239202  33.985678 -118.264977    12
3  06037404504  34.110492 -117.903340     4
4  06037237600  33.978513 -118.283872    12


In [6]:
# --- 5. Aggregate and inspect ---

counts = qct.groupby("PREC").size().rename("qct_count").reset_index()

# Include all precincts (0 for those with no QCT tracts)
all_prec = prec[["PREC", "DIVISION"]].copy()
result = all_prec.merge(counts, on="PREC", how="left").fillna({"qct_count": 0})
result["qct_count"] = result["qct_count"].astype(int)

print(result.sort_values("qct_count", ascending=False).to_string(index=False))

 PREC         DIVISION  qct_count
    4       HOLLENBECK         97
   18        SOUTHEAST         76
   13           NEWTON         53
    5           HARBOR         48
   20          OLYMPIC         48
   16         FOOTHILL         47
   12      77TH STREET         45
    3        SOUTHWEST         25
    2          RAMPART         23
    9         VAN NUYS         22
    6        HOLLYWOOD         19
    7         WILSHIRE         16
   11        NORTHEAST         14
    1          CENTRAL         11
   15  NORTH HOLLYWOOD         10
   19          MISSION         10
   14          PACIFIC          6
   21          TOPANGA          5
   10      WEST VALLEY          4
    8 WEST LOS ANGELES          3
   17       DEVONSHIRE          2


In [7]:
# --- 6. Save output ---

result[["PREC", "qct_count"]].to_csv(OUT_PATH, index=False)
print(f"Saved {OUT_PATH}")

Saved data/census_indicators/qct_by_prec.csv
